In [1]:
import os
from bs4 import BeautifulSoup
import requests
import time
from datetime import datetime
import json
from zipfile import ZipFile
import pandas as pd
import shutil
import zipfile
import random

### Define PATH constant

In [2]:
BASE_PATH = "data"
HTML_PATH = BASE_PATH + "/html"
USER_PATH = BASE_PATH + "/users"

# 1. Get Clubs IDs

In [3]:
def get_number(string):
    return int(string.strip().replace(",", ""))


clubs_id = set()
possibles_users = 0
page = 1

while True:
    print(f"\r{page}", end="")

    time.sleep(3) 
    data = requests.get(f"https://myanimelist.net/clubs.php?p={page}")
    soup = BeautifulSoup(data.text, "html.parser")
    rows = soup.find_all("tr", {"class": "table-data"})
    for row in rows:
        members = get_number(row.find("td", {"class": "ac"}).text)
        club_id = get_number(
            row.find("a", {"class": "fw-b"}).get("href").split("=")[-1]
        )
        if (
            club_id not in clubs_id and members > 30
        ):  # Only save groups with more than 30 members
            possibles_users += members
            clubs_id.add(club_id)

    page += 1
    if possibles_users > 1000000:  # Threshold to stop
        break

with open(f"{BASE_PATH}/clubs.txt", "w") as file:
    for club in clubs_id:
        file.write(f"{club}\n")

22

# 2. Get usernames in every clubs

In [3]:
if not os.path.exists(f"{BASE_PATH}/users_list.txt"):
    with open(f"{BASE_PATH}/users_list.txt", "w", encoding="UTF-8") as file:
        pass
    
if not os.path.exists(f"{BASE_PATH}/_revised_clubs.txt"):
    with open(f"{BASE_PATH}/_revised_clubs.txt", "w", encoding="UTF-8") as file:
        pass

In [4]:
with open(f"{BASE_PATH}/clubs.txt") as file:
    clubs_id = [x.strip() for x in file.readlines()]

with open(f"{BASE_PATH}/users_list.txt", encoding="UTF-8") as file:
    users = set([x.strip() for x in file.readlines()])

with open(f"{BASE_PATH}/_revised_clubs.txt", encoding="UTF-8") as file:
    revised_clubs = set([int(x.strip()) for x in file.readlines()])

len(users), len(revised_clubs), len(clubs_id)

(32070, 184, 702)

In [6]:
%load_ext jupyternotify

<IPython.core.display.Javascript object>

In [8]:

for i, club_id in enumerate(clubs_id):
    if club_id in revised_clubs:
        continue

    page = 1
    while True:
        print(f"\r{i+1}/{len(clubs_id)} --> {str(page).zfill(2)}", end="")
        link = f"https://api.jikan.moe/v3/club/{club_id}/members/{page}"

        try:
            time.sleep(4.2)
            data = requests.get(link)
        except KeyboardInterrupt:
            raise KeyboardInterrupt()
        except: 
            time.sleep(120)
            continue

        if data.status_code != 200:
            break

        with open(f"{BASE_PATH}/users_list.txt", "a", encoding="UTF-8") as file:
            for user in map(lambda x: x["username"], json.loads(data.text)["members"]):
                if user not in users and user != "":
                    file.write(f"{user}\n")
                    users.add(user)
        page += 1

    revised_clubs.add(club_id)
    with open(f"{BASE_PATH}/_revised_clubs.txt", "a", encoding="UTF-8") as file:
        file.write(f"{club_id}\n")

1/702 --> 01

KeyboardInterrupt: 

In [ ]:
temp = users

In [ ]:
import time
import requests
import json


for i, club_id in enumerate(clubs_id):
    if club_id in revised_clubs:
        continue

    page = 1
    while True:
        print(f"\r{i+1}/{len(clubs_id)} --> {str(page).zfill(2)}", end="")
        link = f"https://api.jikan.moe/v4/clubs/{club_id}/members?page={page}"

        try:
            time.sleep(4.2)  # Rate limiting
            response = requests.get(link)
            response.raise_for_status()  # Raise an error for bad responses
        except KeyboardInterrupt:
            raise KeyboardInterrupt()
        except requests.exceptions.RequestException as e: 
            print(f"Request failed: {e}")
            time.sleep(120)  # Wait before retrying
            continue

        # Load the JSON response
        data = response.json()
        members = data.get("data", [])
        
        # If no members are returned, exit the loop
        if not members:
            break

        # Extract usernames from members
        with open(f"{BASE_PATH}/users_list.txt", "a", encoding="UTF-8") as file:
            for user in members:
                username = user.get("username")
                if username and username not in users:
                    file.write(f"{username}\n")
                    users.add(username)
                    
        # Check if there is a next page
        pagination = data.get("pagination", {})
        if not pagination.get("has_next_page", False):
            break  # Exit the loop if there's no next page

        page += 1

    revised_clubs.add(club_id)
    with open(f"{BASE_PATH}/_revised_clubs.txt", "a", encoding="UTF-8") as file:
        file.write(f"{club_id}\n")


In [ ]:

        print(f"\r{i+1}/{len(clubs_id)} --> {str(page).zfill(2)}", end="")
        link = f"https://api.jikan.moe/v4/clubs/84441/members?page=2"

        try:
            data = requests.get(link)
        except KeyboardInterrupt:
            raise KeyboardInterrupt()
        except: 
            time.sleep(120)
 


In [14]:
for f in data:
    print(f)

b'{"status":410,"type":"BadResponseException","message":"v3 has been discontinued. For more information visit https://bit.ly/jikan'
b'-v3-deprecation","upgrade":"Please upgrade to v4: https://docs.api.jikan.moe/"}'


In [6]:
with open(f"{BASE_PATH}/users_list.txt", encoding="UTF-8") as file:
    users = list(set([x.strip() for x in file.readlines()]))[1:]
    random.shuffle(users)

with open(f"{BASE_PATH}/users.csv", "w", encoding="UTF-8") as file:
    file.write("user_id,username\n")
    for i, user in enumerate(users):
        file.write(f"{i},{user}\n")

# 3. Get animelist per user

In [7]:
with open(f"{BASE_PATH}/users.csv", "r", encoding="UTF-8") as file:
    file.readline()
    users = [x.strip().split(",") for x in file.readlines()]
    users = [(int(x[0]), x[1]) for x in users]

last_revised_users = -1
if os.path.exists(f"{BASE_PATH}/_last_revised_users.txt"):
    with open(f"{BASE_PATH}/_last_revised_users.txt", "r", encoding="UTF-8") as file:
        last_revised_users = int(file.readline())

len(users), last_revised_users

(32069, 3781)

In [5]:
%load_ext jupyternotify

<IPython.core.display.Javascript object>

In [8]:

for i, (user_id, username) in enumerate(users):
    if user_id <= last_revised_users:
        continue

    now = datetime.now()
    print(f'\r{str(now).split(".")[0]} --> {i+1}/{len(users)}', end="")
    page = 1
    all_animes = []

    while True:
        link = f"https://api.jikan.moe/v3/user/{username}/animelist/all?page={page}"
        try:
            time.sleep(4.2)
            data = requests.get(link, timeout=15)
        except KeyboardInterrupt:
            raise KeyboardInterrupt()
        except:  
            time.sleep(120)
            continue

        if data.status_code != 200:
            break

        data = json.loads(data.text)
        for anime in data["anime"]:
            all_animes.append((anime["mal_id"], anime["score"], anime["watching_status"], anime["watched_episodes"]))

        page += 1
        if len(data["anime"]) < 300:
            break

    if len(all_animes) != 0:
        with open(f"{USER_PATH}/{user_id}.csv", "w") as f1:
            f1.write(f"anime_id,score,watching_status,watched_episodes\n")
            for anime_id, anime_score, watching_status, watched_episodes in all_animes:
                f1.write(
                    f"{anime_id},{anime_score},{watching_status},{watched_episodes}\n"
                )

    revised_users = user_id
    with open(f"{BASE_PATH}/_last_revised_users.txt", "w", encoding="UTF-8") as file:
        file.write(f"{user_id}\n")

2026-03-10 00:14:41 --> 17565/32069

In [10]:
import time
import requests
import json
from datetime import datetime

# Define your MAL client ID and required fields
X_MAL_CLIENT_ID = "07583d4579b7f663d5589edf375b5a7d"
api_fields = 'alternative_titles,num_episodes,start_date,end_date,my_list_status{comments,start_date,finish_date,num_times_rewatched},list_status{comments,start_date,finish_date,num_times_rewatched},synopsis,popularity,num_list_users,num_scoring_users,nsfw,genres,created_at,updated_at,status,start_season,broadcast,source,average_episode_duration,rating'

headers = {
    "X-MAL-CLIENT-ID": X_MAL_CLIENT_ID
}

# Assuming BASE_PATH, USER_PATH, users, last_revised_users are defined earlier

for i, (user_id, username) in enumerate(users):
    if username != "PlasmaFists":
        continue
    if user_id <= last_revised_users:
        continue

    now = datetime.now()
    print(f'\r{str(now).split(".")[0]} --> {i+1}/{len(users)}', end="")
    offset = 0
    all_animes = []

    while True:
        link = f"https://api.myanimelist.net/v2/users/PlasmaFists/animelist"
        params = {
            "fields": api_fields,
            "limit": 100,
            "offset": offset
        }

        try:
            time.sleep(4)  # Respect rate limits
            response = requests.get(link, headers=headers, params=params, timeout=15)
            response.raise_for_status()  # Raise exception for HTTP errors
        except KeyboardInterrupt:
            raise
        except requests.exceptions.RequestException as e:
            print(f"Request failed for {username}: {e}")
            break

        data = response.json()
        anime_list = data.get("data", [])
        
        if not anime_list:
            break

        # Extract relevant information for each anime
        for anime_entry in anime_list:
            anime = anime_entry["node"]
            anime_status = anime_entry["list_status"]

            all_animes.append((
                anime["id"],
                anime_status.get("score", ""),
                anime_status.get("status", ""),
                anime_status.get("num_watched_episodes", "")
            ))

        # Check if there's a next page
        if not data.get("paging", {}).get("next"):
            break

        offset += 100  # Move to the next batch of results

    # Save the anime list to a CSV file if there are animes
    if all_animes:
        with open(f"{USER_PATH}/{user_id}.csv", "w") as f1:
            f1.write("anime_id,score,watching_status,watched_episodes\n")
            for anime_id, anime_score, watching_status, watched_episodes in all_animes:
                f1.write(f"{anime_id},{anime_score},{watching_status},{watched_episodes}\n")

    # Update the last revised user
    revised_users = user_id
    with open(f"{BASE_PATH}/_last_revised_users.txt", "w", encoding="UTF-8") as file:
        file.write(f"{user_id}\n")


2026-03-10 00:16:06 --> 17565/32069

In [12]:
import requests
import time

# Your MAL Client ID and fields
X_MAL_CLIENT_ID = "e6bcb1517adb2afe8befbf0d8e3eac26"
api_fields = 'alternative_titles,num_episodes,start_date,end_date,my_list_status{comments,start_date,finish_date,num_times_rewatched},list_status{comments,start_date,finish_date,num_times_rewatched},synopsis,popularity,num_list_users,num_scoring_users,nsfw,genres,created_at,updated_at,status,start_season,broadcast,source,average_episode_duration,rating'

headers = {
    "X-MAL-CLIENT-ID": X_MAL_CLIENT_ID
}

# Base URL for user animelist
url = "https://api.myanimelist.net/v2/users/PlasmaFists/animelist"
anime_names = []
page = 1

while True:
    params = {
        "fields": api_fields,  # Include all desired fields
        "limit": 100,          # Max limit per MAL API request
        "offset": (page - 1) * 100
    }

    try:
        response = requests.get(url, headers=headers, params=params)
        response.raise_for_status()  # Raise error for bad responses
    except KeyboardInterrupt:
        raise
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        break

    data = response.json()
    anime_list = data.get("data", [])
    
    # Extract and store anime names
    for anime_entry in anime_list:
        anime_names.append(anime_entry["node"]["title"])  # Get anime title from "node"

    # Check for more pages
    if not data.get("paging", {}).get("next"):
        break  # Exit if there's no next page

    page += 1  # Move to the next page
    time.sleep(2)  # Respect rate limits

# Print all collected anime names
for anime in anime_names:
    print(anime)


100-man no Inochi no Ue ni Ore wa Tatteiru
3-gatsu no Lion
3-gatsu no Lion 2nd Season
3D Kanojo: Real Girl
5-toubun no Hanayome
5-toubun no Hanayome ∬
5-toubun no Hanayome Movie
86
86 Part 2
91 Days
Aharen-san wa Hakarenai
Ajin
Akagami no Shirayuki-hime
Akame ga Kill!
Akatsuki no Yona
Akudama Drive
Akuyaku Reijou Level 99: Watashi wa Ura-Boss desu ga Maou dewa Arimasen
Akuyaku Reijou nanode Last Boss wo Kattemimashita
Amagami SS
Amagi Brilliant Park
Ameku Takao no Suiri Karte
Angel Beats!
Anne Shirley
Another
Ansatsu Kyoushitsu
Ansatsu Kyoushitsu 2nd Season
Ao Ashi
Ao Haru Ride
Ao no Exorcist
Ao no Hako
Araburu Kisetsu no Otome-domo yo.
Arakawa Under the Bridge
Arcana Famiglia
Arifureta Shokugyou de Sekai Saikyou
B: The Beginning
Baby Steps
Baby Steps 2nd Season
Baccano!
Bakemonogatari
Bakuten Shoot Beyblade 2002
Ballpark de Tsukamaete!
Beck
Benriya Saitou-san, Isekai ni Iku
Big Order (TV)
Black Bullet
Black Clover
Black Lagoon
Black Lagoon: Roberta's Blood Trail
Black Lagoon: The Seco

In [13]:
users

[(0, 'letmewatchhentai'),
 (1, 'SifoneDiAnima'),
 (2, 'nanisuki'),
 (3, 'panguboy'),
 (4, 'Takaflame'),
 (5, 'briee_lox23'),
 (6, 'Suzanne83'),
 (7, 'LelmasterxD'),
 (8, 'monsieurcooler'),
 (9, 'SsjRav'),
 (10, 'Tiaaan'),
 (11, 'Keinx'),
 (12, 'Furious16'),
 (13, 'ryuucchii'),
 (14, 'Karakter'),
 (15, 'kathryn123'),
 (16, 'demondevil08'),
 (17, 'gunsandknifes'),
 (18, 'Laboon'),
 (19, 'bennnzzzyyy'),
 (20, 'Esan_chan'),
 (21, 'ZavionX25'),
 (22, 'Kazman'),
 (23, 'KanokSD'),
 (24, 'RSmoke'),
 (25, 'Andu_12'),
 (26, 'Jani442'),
 (27, 'kaine_01-me'),
 (28, 'sunlightangels'),
 (29, 'Zacfury'),
 (30, 'Kei-sama'),
 (31, 'Awwwstin'),
 (32, 'FappingOnly'),
 (33, 'CrazyLarry'),
 (34, 'naddu'),
 (35, 'xkarinx'),
 (36, 'ViniViniVini'),
 (37, 'tehcode'),
 (38, 'Znorax'),
 (39, 'kfagi'),
 (40, 'Skogkatt'),
 (41, 'teamliquidfan'),
 (42, 'Baragannn'),
 (43, 'voofed'),
 (44, 'MrInVerTer'),
 (45, 'jack4s'),
 (46, 'Shion-chama'),
 (47, 'Kidwoohda'),
 (48, 'Tino_StickFolix'),
 (49, 'Gold_Time'),
 (50, 'M

In [14]:
import os
import time
import requests
import json
from datetime import datetime

# Define your MAL client ID and required fields
X_MAL_CLIENT_ID = "07583d4579b7f663d5589edf375b5a7d"

headers = {
    "X-MAL-CLIENT-ID": X_MAL_CLIENT_ID
}

api_fields = 'alternative_titles,num_episodes,start_date,end_date,my_list_status{comments,start_date,finish_date,num_times_rewatched},list_status{comments,start_date,finish_date,num_times_rewatched},synopsis,popularity,num_list_users,num_scoring_users,nsfw,genres,created_at,updated_at,status,start_season,broadcast,source,average_episode_duration,rating'

with open(f"{BASE_PATH}/_last_revised_users.txt", "r", encoding="UTF-8") as file:
    last_revised_user = int(file.read().strip())

for i, (user_id, username) in enumerate(users):
    if user_id <= last_revised_user:
        continue
    print("\n"+username)
    now = datetime.now()
    print(f'\r{str(now).split(".")[0]} --> {i+1}/{len(users)}', end="")
    offset = 0
    limit = 100  # Number of items to retrieve per page
    all_animes = []

    while True:
        link = f"https://api.myanimelist.net/v2/users/{username}/animelist"
        params = {
            "fields": api_fields,
            "limit": limit,
            "offset": offset
        }
        
        try:
            time.sleep(4)  # Respect MAL API rate limits
            response = requests.get(link, headers=headers, params=params, timeout=15)
            response.raise_for_status()  # Raise exception for HTTP errors
        except KeyboardInterrupt:
            raise
        except requests.exceptions.RequestException as e:
            print(f"Request failed for {username}: {e}")
            break

        data = response.json()
        
        # Extract anime list from the 'data' key
        anime_list = data.get("data", [])
        if not anime_list:
            break  # Stop if no more anime data

        # Extract relevant information for each anime
        for anime_entry in anime_list:
            anime = anime_entry["node"]  # Access the anime details
            anime_status = anime_entry["list_status"]  # Access user's anime watching status

            all_animes.append((
                anime["id"], 
                anime_status.get("score", ""), 
                anime_status.get("status", ""), 
                anime_status.get("num_watched_episodes", "")
            ))

        # Update offset for the next batch
        offset += limit

        # Check if there's a next page using pagination
        if not data.get("paging", {}).get("next"):
            break  # Exit if there's no next page

    # Save the anime list to a CSV file if there are animes
    if all_animes:
        os.makedirs(USER_PATH, exist_ok=True)  # Ensure directory exists
        with open(f"{USER_PATH}/{user_id}.csv", "w") as f1:
            f1.write(f"anime_id,score,watching_status,watched_episodes\n")
            for anime_id, anime_score, watching_status, watched_episodes in all_animes:
                f1.write(f"{anime_id},{anime_score},{watching_status},{watched_episodes}\n")

    # Update the last revised user
    revised_users = user_id
    with open(f"{BASE_PATH}/_last_revised_users.txt", "w", encoding="UTF-8") as file:
        file.write(f"{user_id}\n")



_haru_
2026-03-07 23:42:53 --> 4010/32069
decimo10
2026-03-07 23:42:58 --> 4011/32069
KaiqueCW
2026-03-07 23:43:03 --> 4012/32069

KeyboardInterrupt: 

In [10]:
USER_PATH

'data/users'

# 4. Download Anime HTML

In [ ]:
unique_anime = set()
folder = os.listdir(USER_PATH)
for i, user_file in enumerate(folder):
    if ".csv" not in user_file:
        continue

    print(f"\r{i + 1}/{len(folder)}", end="")
    with open(f"{USER_PATH}/{user_file}", "r") as file:
        file.readline()
        for line in file:
            anime = line.strip().split(",")[0]
            unique_anime.add(anime)

print("         ")
print(len(unique_anime))

In [ ]:
MAX = 7  # MAX SECOND TO WAIT PER REQUEST
MIN = 4  # MIN SECONDS TO WAIT PER REQUEST


def sleep():
    time_to_sleep = random.random() * (MAX - MIN) + MIN
    time.sleep(time_to_sleep)


def get_link_by_text(soup, anime_id, text):
    links = list(filter(lambda x: anime_id in x["href"], soup.find_all("a", text=text)))
    return links[0]["href"]


def save(path, data):
    with open(path, "w", encoding="UTF-8") as file:
        file.write(data)


def save_link(link, anime_id, name):
    sleep()
    path = f"{HTML_PATH}/{anime_id}/{name}.html"
    data = requests.get(link)
    soup = BeautifulSoup(data.text, "html.parser")
    soup.script.decompose()
    save(path, soup.prettify())
    return soup


def save_reviews(link, anime_id):
    page = 1
    while True:
        sleep()
        actual_link = f"{link}?p={page}"
        data = requests.get(actual_link)
        soup = BeautifulSoup(data.text, "html.parser")
        reviews = soup.find_all("a", text="Overall Rating")
        if len(reviews) == 0:
            break

        path = f"{HTML_PATH}/{anime_id}/reviews_{page}.html"
        soup.script.decompose()
        save(path, soup.prettify())
        page += 1


def scrap_anime(anime_id):
    path = f"{HTML_PATH}/{anime_id}"
    os.makedirs(path, exist_ok=True)
    sleep()
    data = requests.get(f"https://myanimelist.net/anime/{anime_id}")

    anime_info = data.text
    soup = BeautifulSoup(anime_info, "html.parser")
    soup.script.decompose()
    save(f"{HTML_PATH}/{anime_id}/details.html", soup.prettify())

    link_review = get_link_by_text(soup, anime_id, "Reviews")
    link_recomendations = get_link_by_text(soup, anime_id, "Recommendations")
    link_stats = get_link_by_text(soup, anime_id, "Stats")
    link_staff = get_link_by_text(soup, anime_id, "Characters & Staff")
    link_pictures = get_link_by_text(soup, anime_id, "Pictures")

    save_link(link_pictures, anime_id, "pictures")
    save_link(link_staff, anime_id, "staff")
    save_link(link_stats, anime_id, "stats")
    save_link(link_recomendations, anime_id, "recomendations")
    save_reviews(link_review, anime_id)


def zipdir(path, ziph):
    # ziph is zipfile handle
    for root, dirs, files in os.walk(path):
        for file in files:
            ziph.write(
                os.path.join(root, file),
                os.path.relpath(os.path.join(root, file), path),
            )


In [ ]:
%load_ext jupyternotify

In [ ]:
%%notify -m "Anime scrapping finish"

for i, anime_id in enumerate(unique_anime):
    if os.path.isfile(f"{HTML_PATH}/{anime_id}.zip"):
        continue

    print(f"\r{i+1}/{len(unique_anime)}", end="")

    try:
        scrap_anime(anime_id)
    except KeyboardInterrupt:
        break
    except:  # Other exception wait 2 min and try again
        time.sleep(120)
        continue

    path = f"{HTML_PATH}/{anime_id}"
    zipf = zipfile.ZipFile(f"{path}.zip", "w", zipfile.ZIP_DEFLATED)
    zipdir(path, zipf)
    zipf.close()

    shutil.rmtree(path)

# 5. Scrapping Anime info from HTML

In [ ]:
def extract_zip(input_zip):
    input_zip = ZipFile(input_zip)
    return {name: input_zip.read(name) for name in input_zip.namelist()}

KEYS = ['MAL_ID', 'Name', 'Score', 'Genders', 'English name', 'Japanese name', 'Type', 'Episodes',
        'Aired', 'Premiered', 'Producers', 'Licensors', 'Studios', 'Source', 'Duration', 'Rating',
        'Ranked', 'Popularity', 'Members', 'Favorites', 'Watching', 'Completed', 'On-Hold', 'Dropped',
        'Plan to Watch', 'Score-10', 'Score-9', 'Score-8', 'Score-7', 'Score-6', 'Score-5', 'Score-4',
        'Score-3', 'Score-2', 'Score-1']

In [ ]:
def get_name(info):
    return info.find("h1", {"class": "title-name h1_bold_none"}).text.strip()


def get_english_name(info):
    span = info.findAll("span", {"class": "dark_text"})
    return span.parent.text.strip()


def get_table(a_soup):
    return a_soup.find("div", {"class": "po-r js-statistics-info di-ib"})


def get_score(stats):
    score = stats.find("span", {"itemprop": "ratingValue"})
    if score is None:
        return "Unknown"
    return score.text.strip()


def get_gender(sum_info):
    text = ", ".join(
        [x.text.strip() for x in sum_info.findAll("span", {"itemprop": "genre"})]
    )
    return text


def get_description(sum_info):
    return sum_info.find("td", {"class": "borderClass", "width": "225"})


def get_all_stats(soup):
    return soup.find("div", {"id": "horiznav_nav"}).parent.findAll(
        "div", {"class": "spaceit_pad"}
    )


def get_info_anime(anime_id):
    data = extract_zip(f"data/html/{anime_id}.zip")
    anime_info = data["stats.html"].decode()
    soup = BeautifulSoup(anime_info, "html.parser")

    stats = get_table(soup)
    description = get_description(soup)
    anime_info = {key: "Unknown" for key in KEYS}

    anime_info["MAL_ID"] = anime_id
    anime_info["Name"] = get_name(soup)
    anime_info["Score"] = get_score(stats)
    anime_info["Genders"] = get_gender(description)

    for d in description.findAll("span", {"class": "dark_text"}):
        information = [x.strip().replace(" ", " ") for x in d.parent.text.split(":")]
        category, value = information[0], ":".join(information[1:])
        value.replace("\t", "")

        if category in ["Broadcast", "Synonyms", "Genres", "Score", "Status"]:
            continue

        if category in ["Ranked"]:
            value = value.split("\n")[0]
        if category in ["Producers", "Licensors", "Studios"]:
            value = ", ".join([x.strip() for x in value.split(",")])
        if category in ["Ranked", "Popularity"]:
            value = value.replace("#", "")
        if category in ["Members", "Favorites"]:
            value = value.replace(",", "")
        if category in ["English", "Japanese"]:
            category += " name"

        anime_info[category] = value

    # Stats (Watching, Completed, On-Hold, Dropped, Plan to Watch)
    for d in get_all_stats(soup)[:5]:
        category, value = [x.strip().replace(" ", " ") for x in d.text.split(":")]
        value = value.replace(",", "")
        anime_info[category] = value

    # Stast votes per score
    for d in get_all_stats(soup)[6:]:
        score = d.parent.parent.find("td", {"class": "score-label"}).text.strip()
        value = [x.strip().replace(" ", " ") for x in d.text.split("%")][1].strip(
            "(votes)"
        )
        label = f"Score-{score}"
        anime_info[label] = value.strip()

    for key, value in anime_info.items():
        if str(value) in ["?", "None found, add some", "None", "N/A", "Not available"]:
            anime_info[key] = "Unknown"
    return anime_info

In [ ]:
get_info_anime(5)

In [ ]:
# Generate anime.tsv
anime_revised = set()
exist_file = os.path.exists(f"{BASE_PATH}/anime.tsv")

if exist_file:
    # If the file exist, include new data.
    actual_data = pd.read_csv(f"{BASE_PATH}/anime.tsv", sep="\t")
    anime_revised = list(actual_data.MAL_ID.unique())

actual_data.head()
total_data = []
zips = os.listdir(HTML_PATH)
for i, anime in enumerate(zips):
    if not ".zip" in anime:
        continue

    anime_id = int(anime.strip(".zip"))

    if int(anime_id) in anime_revised:
        continue

    print(f"\r{i+1}/{len(zips)} ({anime_id})", end="")

    anime_id = anime.strip(".zip")
    total_data.append(get_info_anime(anime_id))

if len(total_data):
    df = pd.DataFrame.from_dict(total_data)
    df["MAL_ID"] = pd.to_numeric(df["MAL_ID"])
    df = df.sort_values(by="MAL_ID").reset_index(drop=True)

    if exist_file:
        df = (
            pd.concat([actual_data, df]).sort_values(by="MAL_ID").reset_index(drop=True)
        )

else:
    df = actual_data

pd.set_option("display.max_columns", None)
df.head()

In [ ]:
df.to_csv(f"{BASE_PATH}/anime.tsv", index=False, sep="\t", encoding="UTF-8")

# 6. Create rating_complete.csv

This file only contain animes with `watching_status==2`(complete) and have been rated (`score!=0`).

In [ ]:
if not os.path.exists(f"{BASE_PATH}/rating_complete.csv"):
    with open(f"{BASE_PATH}/rating_complete.csv", "w", encoding="UTF-8") as file:
        file.write("user_id,anime_id,rating\n")

In [ ]:
unique_anime = set()
all_users = sorted(os.listdir(USER_PATH), key=lambda x:int(x.split(".")[0]))

with open(f"{BASE_PATH}/rating_complete.csv", "a") as f1:

    for i, user_file in enumerate(all_users):
        if not user_file.endswith(".csv"):
            continue

        print(f"\r{i+1}/{len(all_users)}", end="")

        user_id = user_file.split(".")[0]
        with open(f"{USER_PATH}/{user_file}", "r") as file:
            file.readline()
            for line in file:
                anime_id, score, watching_status, _ = line.strip().split(",")
                if int(watching_status) == 2 and int(score) != 0:
                    f1.write(f"{user_id},{anime_id},{score}\n")

# 7. Unified animelist

In [ ]:
if not os.path.exists(f"{BASE_PATH}/animelist.csv"):
    with open(f"{BASE_PATH}/animelist.csv", "w", encoding="UTF-8") as file:
        file.write("user_id,anime_id,rating,watching_status,watched_episodes\n")

In [ ]:
unique_anime = set()
all_users = sorted(os.listdir(USER_PATH), key=lambda x:int(x.split(".")[0]))

with open(f"{BASE_PATH}/animelist.csv", "a") as f1:

    for i, user_file in enumerate(all_users):
        if not user_file.endswith(".csv"):
            continue

        print(f"\r{i+1}/{len(all_users)}", end="")

        user_id = user_file.split(".")[0]
        with open(f"{USER_PATH}/{user_file}", "r") as file:
            file.readline()
            for line in file:
                anime_id, score, watching_status, watched_episodes = line.strip().split(",")
                f1.write(f"{user_id},{anime_id},{score},{watching_status},{watched_episodes}\n")

In [3]:
import os

def print_folders(path):
    for root, dirs, files in os.walk(path):
        for dir_name in dirs:
            print(os.path.join(root, dir_name))

print_folders('..')  # Prints full paths to all folders


..\backend
..\data
..\frontend
..\backend\.git
..\backend\.vs
..\backend\Kiroku.API
..\backend\Kiroku.Application
..\backend\Kiroku.Data
..\backend\Kiroku.Domain
..\backend\Kiroku.Infrastructure
..\backend\obj
..\backend\.git\hooks
..\backend\.git\info
..\backend\.git\logs
..\backend\.git\objects
..\backend\.git\refs
..\backend\.git\logs\refs
..\backend\.git\logs\refs\heads
..\backend\.git\objects\04
..\backend\.git\objects\0c
..\backend\.git\objects\10
..\backend\.git\objects\12
..\backend\.git\objects\1d
..\backend\.git\objects\1e
..\backend\.git\objects\23
..\backend\.git\objects\2a
..\backend\.git\objects\2d
..\backend\.git\objects\2f
..\backend\.git\objects\31
..\backend\.git\objects\3a
..\backend\.git\objects\3d
..\backend\.git\objects\3e
..\backend\.git\objects\45
..\backend\.git\objects\50
..\backend\.git\objects\5a
..\backend\.git\objects\60
..\backend\.git\objects\6a
..\backend\.git\objects\6d
..\backend\.git\objects\6e
..\backend\.git\objects\75
..\backend\.git\objects\77
..